# Aula 7 — Sinal v2: Vol-Adjusted Momentum

**Intensivão Quant AI — ImpactUFSCar**

---

O sinal v1 mede **quanto** uma ação subiu nos últimos 12 meses. O problema: uma ação que subiu 30% com 80% de volatilidade não é necessariamente um sinal melhor do que uma que subiu 20% com 20% de volatilidade.

Hoje vamos ajustar o sinal pela volatilidade — transformando o momentum bruto num sinal mais parecido com um Sharpe ratio do período de formação.

Ao final, você terá:
- Entendido por que e como normalizar o sinal por vol
- Calculado o sinal v2 e comparado com o v1
- Medido o impacto no turnover e nas métricas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Dados das aulas anteriores
ret_diarios  = pd.read_parquet('retornos_diarios_limpo.parquet')
ret_mensais  = pd.read_parquet('retornos_mensais_limpo.parquet')
sinal_v1     = pd.read_parquet('sinal_v1.parquet')
pesos_v1     = pd.read_parquet('pesos_v1.parquet')
pesos_v2_alo = pd.read_parquet('pesos_v2.parquet')  # esquema de alocação da Aula 6

ibov_raw = yf.download('^BVSP', start='2010-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)['Close'].squeeze()
ret_ibov = np.log(ibov_raw / ibov_raw.shift(1)).resample('ME').sum()

# Funções da Aula 5 (copiadas para auto-contenção)
def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    n_anos    = len(retornos) / 12
    ret_total = np.exp(retornos.sum()) - 1
    cagr      = (1 + ret_total) ** (1 / n_anos) - 1
    vol_anual = retornos.std() * np.sqrt(12)
    excesso   = retornos - rf_mensal
    sharpe    = (excesso.mean() / excesso.std()) * np.sqrt(12)
    ret_neg   = excesso[excesso < 0]
    sortino   = (excesso.mean() * 12) / (ret_neg.std() * np.sqrt(12)) if len(ret_neg) > 1 else np.nan
    valor     = np.exp(retornos.cumsum())
    mdd       = ((valor - valor.cummax()) / valor.cummax()).min()
    calmar    = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta_val = np.nan, np.nan
    if benchmark is not None:
        idx = retornos.index.intersection(benchmark.index)
        if len(idx) > 10:
            sl, ic, *_ = linregress(benchmark[idx], retornos[idx])
            beta_val, alpha = sl, ic * 12
    return dict(nome=nome, cagr=cagr, vol_anual=vol_anual, sharpe=sharpe,
                sortino=sortino, max_drawdown=mdd, calmar=calmar,
                alpha_anual=alpha, beta=beta_val)

def exibir_metricas(*ms):
    fmt = dict(cagr='{:.1%}', vol_anual='{:.1%}', sharpe='{:.2f}',
               sortino='{:.2f}', max_drawdown='{:.1%}', calmar='{:.2f}',
               alpha_anual='{:.2%}', beta='{:.2f}')
    labels = dict(cagr='CAGR', vol_anual='Vol Anual', sharpe='Sharpe',
                  sortino='Sortino', max_drawdown='Max Drawdown',
                  calmar='Calmar', alpha_anual='Alpha', beta='Beta')
    nomes = [d['nome'] for d in ms]
    w = max(14, max(len(n) for n in nomes) + 2)
    h = f"{'Métrica':<22}" + ''.join(f"{n:>{w}}" for n in nomes)
    print(h); print('─' * len(h))
    for k, lb in labels.items():
        linha = f"{lb:<22}"
        for d in ms:
            v = d.get(k, np.nan)
            s = fmt[k].format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'
            linha += f"{s:>{w}}"
        print(linha)

N_ATIVOS = 10
print("Setup concluído.")

## 1. O problema do sinal bruto

O sinal v1 mede retorno acumulado — mas retorno sem levar risco em conta é incompleto.

Compare dois ativos:

| Ativo | Retorno 12-1 | Volatilidade | Sinal v1 | "Qualidade" |
|---|---|---|---|---|
| A | +30% | 70% ao ano | Alto | Baixa (só ganhou muito porque é muito volátil) |
| B | +20% | 15% ao ano | Médio | Alta (ganhou bem com risco controlado) |

O sinal v1 prefere A. O sinal v2 prefere B.

**A ideia:** dividir o retorno de formação pela volatilidade realizada — transformando o momentum num **Sharpe ratio do período de formação**:

$$\text{sinal}_{v2} = \frac{\text{retorno}_{12-1}}{\sigma_{\text{rolling}}}$$

Isso normaliza o sinal pelo risco — ativos que entregaram retorno de forma mais consistente ganham ranks maiores.

## 2. Volatilidade — a janela certa

Na Aula 3 vimos que a volatilidade muda ao longo do tempo. A escolha da janela rolling para calcular a vol de normalização importa:

| Janela | Comportamento | Indicado para |
|---|---|---|
| 21 dias (1 mês) | Reage rápido a choques, muito ruidosa | Estratégias de alta frequência |
| 63 dias (3 meses) | Balanceamento entre reatividade e estabilidade | **Nossa escolha** |
| 252 dias (1 ano) | Muito estável, lenta para reagir | Análise de longo prazo |

Usamos **63 dias** — janela padrão em estratégias de momentum. Ela captura o regime de volatilidade recente sem ser excessivamente ruidosa.

In [ ]:
# Volatilidade rolling 63 dias — calculada sobre retornos diários e convertida para mensal
# Anualizada: vol_diaria * sqrt(252)
# Para usar na mesma escala que o sinal (retorno de 11 meses), mantemos anualizada

JANELA_VOL = 63

# Calcula vol diária rolling e reamostra para frequência mensal (última obs. de cada mês)
vol_rolling_diaria  = ret_diarios.rolling(JANELA_VOL).std() * np.sqrt(252)
vol_rolling_mensal  = vol_rolling_diaria.resample('ME').last()

# Alinha com o índice do sinal
vol_rolling_mensal  = vol_rolling_mensal.reindex(sinal_v1.index)

print(f"Vol rolling ({JANELA_VOL}d) — shape: {vol_rolling_mensal.shape}")
print(f"\nVol rolling de PETR4 e WEGE3 (últimos 6 meses):")
print(vol_rolling_mensal[['PETR4.SA', 'WEGE3.SA']].tail(6)
      .applymap(lambda x: f'{x:.1%}' if pd.notna(x) else 'N/A'))

In [ ]:
# Visualização: vol rolling de 4 ações ao longo do tempo
acoes_plot = ['PETR4.SA', 'VALE3.SA', 'WEGE3.SA', 'BBAS3.SA']
acoes_plot = [a for a in acoes_plot if a in vol_rolling_mensal.columns]

fig, ax = plt.subplots(figsize=(13, 5))
for acao in acoes_plot:
    vol_rolling_mensal[acao].plot(ax=ax, label=acao.replace('.SA', ''), alpha=0.8)

ax.set_title(f'Volatilidade rolling {JANELA_VOL} dias (anualizada)')
ax.set_ylabel('Vol anualizada')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 3. Calculando o Sinal v2

$$\text{sinal}_{v2,i,t} = \frac{\text{sinal}_{v1,i,t}}{\sigma_{i,t}^{\text{rolling}}}$$

Onde $\sigma_{i,t}^{\text{rolling}}$ é a volatilidade anualizada dos últimos 63 dias de ação $i$ até o mês $t$.

**Atenção ao alinhamento:** o sinal v1 já usa dados até $t-2$. A volatilidade rolling também é calculada com dados disponíveis até o momento do rebalanceamento — sem look-ahead.

In [ ]:
# Sinal v2 = sinal_v1 / vol_rolling
# Substituímos vol = 0 ou NaN por NaN para evitar divisão por zero
vol_safe   = vol_rolling_mensal.replace(0, np.nan)
sinal_v2   = sinal_v1 / vol_safe

print(f"Sinal v2 calculado — shape: {sinal_v2.shape}")
print(f"NaNs no v2 vs v1:")
print(f"  v1: {sinal_v1.isna().sum().sum():,} células")
print(f"  v2: {sinal_v2.isna().sum().sum():,} células  "
      f"(a mais = meses sem vol calculada)")

In [ ]:
# Comparação: sinal v1 vs v2 para um mês específico
data_ex = sinal_v2.dropna(how='all').index[-1]

comp = pd.DataFrame({
    'sinal_v1': sinal_v1.loc[data_ex],
    'sinal_v2': sinal_v2.loc[data_ex],
    'vol':      vol_rolling_mensal.loc[data_ex],
}).dropna()

comp['rank_v1'] = comp['sinal_v1'].rank(ascending=False)
comp['rank_v2'] = comp['sinal_v2'].rank(ascending=False)
comp['delta_rank'] = comp['rank_v1'] - comp['rank_v2']
comp.index = comp.index.str.replace('.SA', '')

# Ativos que sobem muito no ranking ao mudar para v2 (alta vol, descartados pelo ajuste)
print(f"Mês: {data_ex.strftime('%b/%Y')}")
print("\nAtivos que CAEM mais no ranking (v1→v2) — alta vol penaliza:")
print(comp.nlargest(5, 'delta_rank')[['sinal_v1','vol','sinal_v2','rank_v1','rank_v2']]
      .to_string(float_format='{:.2f}'.format))

print("\nAtivos que SOBEM mais no ranking (v1→v2) — baixa vol beneficia:")
print(comp.nsmallest(5, 'delta_rank')[['sinal_v1','vol','sinal_v2','rank_v1','rank_v2']]
      .to_string(float_format='{:.2f}'.format))

In [ ]:
# Scatter: rank v1 vs rank v2 — onde diferem?
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(comp['rank_v1'], comp['rank_v2'],
           c=comp['vol'], cmap='RdYlGn_r', s=60, alpha=0.8)

# Linha de identidade (mesmo rank nos dois)
lim = max(comp['rank_v1'].max(), comp['rank_v2'].max())
ax.plot([1, lim], [1, lim], 'k--', linewidth=1, label='rank idêntico')

# Anota pontos com maior delta
for _, row in comp.nlargest(3, 'delta_rank').iterrows():
    ax.annotate(_, (row['rank_v1'], row['rank_v2']),
                xytext=(5, 5), textcoords='offset points', fontsize=8, color='red')

sm = plt.cm.ScalarMappable(cmap='RdYlGn_r',
     norm=plt.Normalize(comp['vol'].min(), comp['vol'].max()))
plt.colorbar(sm, ax=ax, label='Volatilidade')
ax.set_xlabel('Rank no Sinal v1 (1 = melhor)')
ax.set_ylabel('Rank no Sinal v2 (1 = melhor)')
ax.set_title('Impacto do ajuste por vol no ranking')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Construindo a carteira com Sinal v2

Usamos o mesmo esquema de alocação da Aula 6 (signal-weighted), mas com o ranking derivado do sinal v2.

In [ ]:
ranking_v2 = sinal_v2.rank(axis=1, ascending=True, pct=True)

def calcular_pesos_signal_weighted(ranking_row, N=10):
    validos = ranking_row.dropna()
    if len(validos) < N:
        return pd.Series(0.0, index=ranking_row.index)
    top_n  = validos.nlargest(N)
    pesos  = pd.Series(0.0, index=ranking_row.index)
    pesos[top_n.index] = top_n / top_n.sum()
    return pesos

# Pesos v1 (sinal bruto) e v2 (vol-adjusted) — ambos signal-weighted
ranking_v1 = sinal_v1.rank(axis=1, ascending=True, pct=True)
pesos_s_v1 = ranking_v1.apply(calcular_pesos_signal_weighted, axis=1, N=N_ATIVOS)
pesos_s_v2 = ranking_v2.apply(calcular_pesos_signal_weighted, axis=1, N=N_ATIVOS)

print(f"Pesos v1 calculados: {(pesos_s_v1.sum(axis=1) > 0).sum()} meses")
print(f"Pesos v2 calculados: {(pesos_s_v2.sum(axis=1) > 0).sum()} meses")

## 5. Turnover — o custo invisível

**Turnover** é a fração da carteira que é substituída em cada rebalanceamento.

$$\text{Turnover}_t = \frac{1}{2} \sum_{i} |w_{i,t} - w_{i,t-1}^{\text{drift}}|$$

Onde $w_{i,t-1}^{\text{drift}}$ é o peso do ativo antes do rebalanceamento (depois da variação de preços).

Por que importa? Cada compra e venda tem custo:
- **Corretagem** — taxa por operação
- **Spread bid-ask** — diferença entre preço de compra e venda
- **Impacto de mercado** — seu próprio volume move o preço contra você

Uma estratégia com 50% de turnover mensal e 0.3% de custo por operação perde ~3.6% ao ano só em custos. Isso pode eliminar completamente o alpha de uma estratégia marginal.

O sinal v2 pode ter turnover diferente do v1 porque seleciona ativos diferentes.

In [ ]:
def calcular_turnover(pesos_df):
    """Turnover mensal: % da carteira que muda a cada rebalanceamento."""
    # Diferença absoluta de pesos entre meses consecutivos
    delta = pesos_df.diff().abs().sum(axis=1) / 2
    # Exclui o primeiro mês (sem referência anterior) e meses sem carteira
    meses_validos = pesos_df.sum(axis=1) > 0
    return delta[meses_validos].iloc[1:]

to_v1 = calcular_turnover(pesos_s_v1)
to_v2 = calcular_turnover(pesos_s_v2)

fig, ax = plt.subplots(figsize=(13, 4))
to_v1.plot(ax=ax, label=f'Sinal v1 — média {to_v1.mean():.1%}',
           alpha=0.7, color='steelblue')
to_v2.plot(ax=ax, label=f'Sinal v2 — média {to_v2.mean():.1%}',
           alpha=0.7, color='darkorange')
ax.axhline(to_v1.mean(), color='steelblue', linestyle='--', linewidth=1)
ax.axhline(to_v2.mean(), color='darkorange', linestyle='--', linewidth=1)
ax.set_title('Turnover mensal — Sinal v1 vs v2')
ax.set_ylabel('Turnover')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f"Turnover médio — v1: {to_v1.mean():.1%} | v2: {to_v2.mean():.1%}")

## 6. Comparação: Sinal v1 vs v2

In [ ]:
def retorno_carteira(pesos_df, ret_mensais):
    ret = (pesos_df * ret_mensais).sum(axis=1)
    return ret[pesos_df.sum(axis=1) > 0]

ret_v1 = retorno_carteira(pesos_s_v1, ret_mensais)
ret_v2 = retorno_carteira(pesos_s_v2, ret_mensais)

idx = ret_v1.index.intersection(ret_v2.index)
ret_bm = ret_ibov.reindex(idx).dropna()
ret_v1 = ret_v1.reindex(ret_bm.index)
ret_v2 = ret_v2.reindex(ret_bm.index)

# Curvas acumuladas
fig, ax = plt.subplots(figsize=(13, 5))
np.exp(ret_v1.cumsum()).plot(ax=ax, label='Sinal v1 (bruto)', linewidth=2)
np.exp(ret_v2.cumsum()).plot(ax=ax, label='Sinal v2 (vol-adjusted)', linewidth=2)
np.exp(ret_bm.cumsum()).plot(ax=ax, label='IBOVESPA',
                              linewidth=1.5, linestyle='--', color='gray')
ax.set_title('Retorno acumulado — Sinal v1 vs Sinal v2')
ax.set_ylabel('Valor (base 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

# Diferença de retorno mês a mês
diff = ret_v2 - ret_v1
fig, ax = plt.subplots(figsize=(13, 3))
cores = ['steelblue' if d >= 0 else 'tomato' for d in diff]
ax.bar(diff.index, diff, width=20, color=cores, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title('Diferença mensal: v2 − v1 (azul = v2 ganhou, vermelho = v1 ganhou)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
plt.tight_layout()
plt.show()

In [ ]:
# Métricas comparativas
m_v1 = calcular_metricas(ret_v1, ret_bm, 'Sinal v1')
m_v2 = calcular_metricas(ret_v2, ret_bm, 'Sinal v2 (vol-adj)')
m_bm = calcular_metricas(ret_bm, nome='IBOVESPA')

print()
exibir_metricas(m_v1, m_v2, m_bm)

## 7. Estabilidade do sinal — consistência ao longo do tempo

Uma métrica importante que a banca avalia: o sinal funciona consistentemente ou só em alguns subperíodos? Dividimos o histórico em janelas de 3 anos e calculamos o Sharpe em cada uma.

In [ ]:
# Sharpe rolling de 36 meses
sharpe_rolling_v1 = ret_v1.rolling(36).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(12) if x.std() > 0 else np.nan
)
sharpe_rolling_v2 = ret_v2.rolling(36).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(12) if x.std() > 0 else np.nan
)

fig, ax = plt.subplots(figsize=(13, 4))
sharpe_rolling_v1.plot(ax=ax, label='Sinal v1', alpha=0.8)
sharpe_rolling_v2.plot(ax=ax, label='Sinal v2', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.5, color='green', linewidth=0.8, linestyle=':', label='Sharpe = 0.5')
ax.set_title('Sharpe rolling 36 meses — estabilidade ao longo do tempo')
ax.set_ylabel('Sharpe anualizado (janela 3 anos)')
ax.legend()
plt.tight_layout()
plt.show()

# % do tempo com Sharpe positivo
for nome, sr in [('v1', sharpe_rolling_v1), ('v2', sharpe_rolling_v2)]:
    pos = (sr.dropna() > 0).mean()
    print(f"Sinal {nome}: Sharpe rolling positivo em {pos:.0%} dos subperíodos")

## 8. Salvando o sinal v2

In [ ]:
sinal_v2.to_parquet('sinal_v2.parquet')
pesos_s_v2.to_parquet('pesos_sinal_v2.parquet')
ret_v2.to_frame('ret_momentum').to_parquet('retorno_carteira_sinal_v2.parquet')

print("Arquivos salvos:")
print("  sinal_v2.parquet")
print("  pesos_sinal_v2.parquet")
print("  retorno_carteira_sinal_v2.parquet")
print()
print("Na Aula 8, você vai adicionar custos de transação e walk-forward")
print("sobre a versão do sinal que performou melhor aqui.")

## Resumo da aula

| Conceito | Implementação |
|---|---|
| Vol-adjusted momentum | `sinal_v1 / vol_rolling_63d` |
| Janela de vol | 63 dias — equilíbrio entre reatividade e estabilidade |
| Por que normalizar | Ativo com mesmo retorno mas menor vol tem sinal mais forte |
| Turnover | `pesos.diff().abs().sum(axis=1) / 2` — mede substituição mensal da carteira |
| Sharpe rolling | Diagnóstica consistência do sinal em subperíodos |

---

## Para replicar em casa

1. Rode o notebook e compare o Sharpe de v1 e v2 nos dados do seu grupo
2. Teste outras janelas de vol: `JANELA_VOL = 21` e `JANELA_VOL = 126` — como muda o resultado?
3. Observe o gráfico de Sharpe rolling: há subperíodos em que a estratégia claramente não funcionou? O que acontecia no mercado nessa época?

---

*ImpactUFSCar — Diretoria de Quant*